## Quantifi DSP Demo - Products and Trades

The purpose of this notebook is to demonstrate **Quantifi Data Science Platform (DSP)** capabilities. 

Presented below are some examples of Quantifi framework integration with various popular Data Science and data manipulation and presentation tools available in Python. In particular, we demonstrate the following aspects of the framework: 

- Integration of Quantifi data and pricing models/objects loaded directly from Quantifi Risk into the Jupyter notebook
- Pricing and risk calculations using Quantifi Toolkit directly in the Python code
- Using standard Python libraries (such as *Pandas* and *MatPlotLib*) with Quantifi



***

### Set up Quantifi Toolkit and DSP Access 

**Import packages**

First, we need to load various libraries that we will use in our notebook:

In [1]:
# Import standard packages
import pandas as pd
import numpy as np
import datetime
import time
import os
import pprint

# Import packages for display formatting/graph plotting
import ipywidgets as widgets
import matplotlib.pyplot as plt

# Import packages Quantifi Toolkit packages 
from Quantifi.Toolkit.Products import Bond
from Quantifi.Toolkit.Pricers import BondPricer
from Quantifi.Toolkit.Base import Dt, DayCount, Frequency, Calendar, QuotingConvention, BumpTarget
from Quantifi.Toolkit.Sensitivity import Sensitivities, Sensitivities2, BumpType, SensitivityMethod
from Quantifi.Toolkit.Curves import CurveUtil

# Import Quantifi Rest API client functions
# Capture namespace before import
dir_pre_qapi_client_import = set(dir())
from qservices.QuantifiApiClient import *
dir_post_qapi_client_import = set(dir())
# Find the new names that were imported
new_names_post_api_import = dir_post_qapi_client_import - dir_pre_qapi_client_import
print("Newly imported names post qapi import:")
pprint.pprint(sorted(new_names_post_api_import))

pd.options.display.float_format = '{:,.1f}'.format






Newly imported names post qapi import:
['Any',
 'Connect',
 'Currency',
 'FXCurveQuotingType',
 'HttpNegotiateAuth',
 'List',
 'QException',
 'Result',
 'SurvivalCurve',
 'Tuple',
 'Union',
 'Util',
 'XmlSerialization',
 'clr',
 'dir_pre_qapi_client_import',
 'environ',
 'json',
 'locate',
 'qservices',
 'requests',
 'reset',
 'sys',
 'urllib']


In [2]:
# Print only public attributes/methods of Util
pprint.pprint([name for name in dir(Util) if not name.startswith('_')])

['get_data_service_url_prefix',
 'get_entity_data_service_url_prefix',
 'get_market_env_service_url_prefix',
 'get_request',
 'get_timeseries_data_service_url_prefix']


In [3]:
# Print only public attributes/methods of Connect
pprint.pprint([name for name in dir(Connect) if not name.startswith('_')])

['data_query',
 'data_query_VaR_losses',
 'data_query_risk_results',
 'find_rate_curves',
 'get_bond_data',
 'get_discount_curve',
 'get_discount_curves',
 'get_fx_curve',
 'get_inflation_curve',
 'get_inflation_curves',
 'get_legal_entities',
 'get_legal_entity_data',
 'get_pricer_by_cusip',
 'get_pricer_by_externalid',
 'get_pricer_by_figi',
 'get_pricer_by_isin',
 'get_pricer_by_security_identifier',
 'get_pricer_by_trade_id',
 'get_pricers_by_cusip',
 'get_pricers_by_externalid',
 'get_pricers_by_figi',
 'get_pricers_by_isin',
 'get_pricers_by_security_identifiers',
 'get_pricers_by_trade_ids',
 'get_products',
 'get_quote_data',
 'get_quotes_data',
 'get_rate_curve',
 'get_rate_curves',
 'get_securities_by_identifier',
 'get_securities_by_identifiers',
 'get_shift_data',
 'get_shifts_data',
 'get_survival_curve',
 'get_survival_curves',
 'get_trade_by_trade_id',
 'get_trades',
 'get_trades_by_trade_ids',
 'get_zspread_implied_cds_curve']


### Set database query risk date

In [4]:
query_risk_date = '2025-06-20'

In [5]:

# Import some sensitivities directly
from Quantifi.Toolkit.Sensitivity.Sensitivities import Rolldown


Rolldown??





Call signature: Rolldown(*arguments, **kw)
Type:           FunctionGroup
String form:    FunctionGroup: Rolldown
Length:         4
File:           c:\users\mmoore\appdata\local\programs\python\python312\lib\site-packages\qn\_clr\overloads.py
Source:        
class FunctionGroup(object):
    """ A group of overloaded functions sharing the same name
    """

    is_function_group = True

    def __init__(self, name, *functions):
        self.__name__ = name
        instance_fn = self.instance_fns = []
        static_fn = self.static_fns = []
        for func in functions:
            if func.is_static():
                static_fn.append(func)
            else:
                instance_fn.append(func)

    def __get__(self, instance, cls):
        if instance is None:
            return self
        return Method(self, instance)

    def append(self, func):
        """ Adds a function to the group """
        if func.is_static():
            self.static_fns.append(func)
        else:
     

**Set Access Parameters**

Next, we define some access parameters for loading data from Quantifi Risk Database:

In [6]:
# Define database access parameters and pricing date

# For locally installed Quantifi Risk
#os.environ["Q_DATA_SERVICE_URL_PREFIX"] = "https://hb-dc1ut-qfidsp:47200"   #UAT URL
os.environ["Q_DATA_SERVICE_URL_PREFIX"] = "https://hb-dc1pd-qfidsp:47200" #Prod URL
#os.environ["Q_MARKET_ENV_SERVICE_URL_PREFIX"] = "https://hb-dc1ut-qfidsp:47201"  #UAT URL
os.environ["Q_MARKET_ENV_SERVICE_URL_PREFIX"] = "https://hb-dc1pd-qfidsp:47201" #Prod URL
os.environ["Q_LOCAL_INSTALL"] = "True"

# Suppress warnings due to unverified HTTPS certificate
requests.packages.urllib3.disable_warnings()

In [7]:
print(Util.get_data_service_url_prefix())
#Util.get_data_service_url_prefix('https://hb-dc1ut-qfidsp:47200') #UAT
Util.get_data_service_url_prefix('https://hb-dc1pd-qfidsp:47200')

https://hb-dc1pd-qfidsp:47200


'https://hb-dc1pd-qfidsp:47200'

***

### Load Static Data From the Database

We can load static data from the database, using HQL or functions specific to some types of data. In this example, we are loading a definition of a bond.

In [8]:
# Load a bond product definition from Quantifi Database
bond_info = Connect.get_bond_data(name = 'DOMC 6.75% Oct28 144A', ds_url="https://hb-dc1pd-qfidsp:47200", verify_https_cert = False)
display(bond_info)

{'$type': 'Bond',
 'ObjectId': '86412817850514944',
 'Name': 'DOMC 6.75% Oct28 144A',
 'Description': 'DOMC 6.75% Oct28 144A',
 'Issue': '2021-10-18',
 'Effective': '2021-10-18',
 'Maturity': '2028-10-01',
 'DaysToSettle': 1,
 'Ccy': 'USD',
 'Notional': 1.0,
 'Tags': [{'$type': 'Tag', 'Name': 'ID_BB_UNIQUE', 'Value': 'COBR6768609'},
  {'$type': 'Tag', 'Name': 'SecurityId', 'Value': 'COBO010501'}],
 'ExternalId': 'BBG012QTV0Y0',
 'Issuer': {'$type': 'LegalEntity',
  'Name': 'Domtar Corp',
  'ObjectId': '28147497671078843'},
 'CallSchedule': [{'$type': 'CallPeriod',
   'StartDate': '2024-10-01',
   'EndDate': '2025-09-30',
   'CallPrice': 1.03375,
   'TriggerPrice': 1.0,
   'Style': 'Bermudan',
   'NotificationDate': '',
   'UnderlyingStartDate': ''},
  {'$type': 'CallPeriod',
   'StartDate': '2025-10-01',
   'EndDate': '2026-09-30',
   'CallPrice': 1.01688,
   'TriggerPrice': 1.0,
   'Style': 'Bermudan',
   'NotificationDate': '',
   'UnderlyingStartDate': ''},
  {'$type': 'CallPeriod',

For the above bond, we can load issuer static data:

In [9]:
# Get Issuer name from the above bond
le_name = bond_info['Issuer']['Name']
print(f"le name: {le_name}\n")

# Load Issuer (Legal Entity) definition from the database
le_info = Connect.get_legal_entity_data(name = le_name, verify_https_cert = False)
display(le_info)

le name: Domtar Corp



{'$type': 'LegalEntity',
 'ObjectId': '28147497671078843',
 'Name': 'Domtar Corp',
 'LongName': 'Domtar Corporation',
 'Ticker': 'DOMC',
 'CLIP': '27CCAE',
 'LEI': 'XYSIJOU3HNVROZWNA008',
 'Tags': [],
 'Country': {'$type': 'Country',
  'ISOAlpha3': 'USA',
  'ObjectId': '55732045388710110'},
 'CreditGrade': 'None',
 'CapitalStructureRatings': {},
 'EntityRoles': ['Reference'],
 'DefaultRestructuring': 'ModifiedModified',
 'RatingActions': [],
 'Sectors': [{'$type': 'SectorItem',
   'Sector': {'$type': 'Sector',
    'SectorType': {'$type': 'SectorType',
     'Name': 'Markit Sector',
     'ObjectId': '38562071809359873'},
    'Name': 'Basic Materials',
    'Value': 0.0,
    'ObjectId': '38843546786070564'}}],
 'CurveCurrency': 'None',
 'CurveSeniority': 'SeniorSecured'}

***

### Loading Pricer Objects from the Risk Engine

We can load pricer objects for products and trades defined in the Quantifi Risk application. The pricer objects integrate static data, market data and a pricing model and provide risk measure calculations for all product types in Quantifi.

This functionality relies on a **Calculation Environment** being set up in the Risk Application. Calculation Environment definition controls the settings for the pricing models and risk calculation, as well as market data used for pricing.  

The following cell loads a **BondPricer** object from the Qauntifi Risk application  and some of its properties. Note that the pricer comes with all of the data (static data, curves, market quotes, etc) which are necessary for pricing:

### CDX Pricer Object

In [10]:
cdx_pricer = Connect.get_pricer_by_trade_id(as_of = query_risk_date, 
                                             trade_id = "CDX0022462", 
                                             calc_env = 'Official', 
                                             verify_https_cert = False)


dir(cdx_pricer)




['<clr_ctor>',
 'AccrualDays',
 'Accrued',
 'AdjustDurationForRemainingNotional',
 'ApproximateForFastCalculation',
 'AsOf',
 'Basis',
 'BasisFactor',
 'BreakEvenPremium',
 'BuildPaymentPricer',
 'CDX',
 'Carry',
 'Cashflow',
 'Clone',
 'CollectValidationErrors',
 'CurrentNotional',
 'CurrentRate',
 'DiscountCurve',
 'DiscountToSettle',
 'EffectiveNotional',
 'Equals',
 'EquivCDSFwdPremium01',
 'EquivCDSRate01',
 'EquivCDSRecovery01',
 'EquivCDSSpread01',
 'EquivCDSSpreadGamma',
 'EquivCDSTheta',
 'EquivalentCDSPricer',
 'ExpectedLoss',
 'Factor',
 'FullPrice',
 'FwdPremium',
 'FwdPremium01',
 'GenerateCashflow',
 'GeneratePayments',
 'GetDerivativesWrtOrdinates',
 'GetHashCode',
 'GetPaymentSchedule',
 'GetType',
 'HasAnalyticDerivatives',
 'ImpliedQuotedSpread',
 'IntrinsicBreakEvenPremium',
 'IntrinsicRiskyDuration',
 'IntrinsicSurvivalProbability',
 'IntrinsicValue',
 'IntrinsicValue_direct',
 'IsTerminated',
 'MTMCarry',
 'MarkToMarketFactor',
 'MarketPremium',
 'MarketPrice',
 'M

In [ ]:
current_notional = cdx_pricer.CurrentNotional
print(f"current_notional:\n{current_notional}") #Not sure what this is
current_rate = cdx_pricer.CurrentRate
print(f"current_rate:\n{current_rate}") #Not sure what this is
effective_notional = cdx_pricer.EffectiveNotional
print(f"effective_notional:\n{effective_notional}")#This is the post-factor notional (-104.1mm * 73/75)


current_notional:
-99934959.0
current_rate:
0.0
effective_notional:
-101323653.0


In [11]:
Rolldown(pricer=cdx_pricer,clean=True)

10417.72071686294

In [12]:
Rolldown(pricer=cdx_pricer,clean=False)

-3655.008866470307

In [13]:
dir(cdx_pricer)

['<clr_ctor>',
 'AccrualDays',
 'Accrued',
 'AdjustDurationForRemainingNotional',
 'ApproximateForFastCalculation',
 'AsOf',
 'Basis',
 'BasisFactor',
 'BreakEvenPremium',
 'BuildPaymentPricer',
 'CDX',
 'Carry',
 'Cashflow',
 'Clone',
 'CollectValidationErrors',
 'CurrentNotional',
 'CurrentRate',
 'DiscountCurve',
 'DiscountToSettle',
 'EffectiveNotional',
 'Equals',
 'EquivCDSFwdPremium01',
 'EquivCDSRate01',
 'EquivCDSRecovery01',
 'EquivCDSSpread01',
 'EquivCDSSpreadGamma',
 'EquivCDSTheta',
 'EquivalentCDSPricer',
 'ExpectedLoss',
 'Factor',
 'FullPrice',
 'FwdPremium',
 'FwdPremium01',
 'GenerateCashflow',
 'GeneratePayments',
 'GetDerivativesWrtOrdinates',
 'GetHashCode',
 'GetPaymentSchedule',
 'GetType',
 'HasAnalyticDerivatives',
 'ImpliedQuotedSpread',
 'IntrinsicBreakEvenPremium',
 'IntrinsicRiskyDuration',
 'IntrinsicSurvivalProbability',
 'IntrinsicValue',
 'IntrinsicValue_direct',
 'IsTerminated',
 'MTMCarry',
 'MarkToMarketFactor',
 'MarketPremium',
 'MarketPrice',
 'M

### SN CDS Pricer Object

In [14]:
sn_pricer = Connect.get_pricer_by_trade_id(as_of = query_risk_date, 
                                             trade_id = "CDS0022423", 
                                             calc_env = 'Official', 
                                             verify_https_cert = False)


dir(sn_pricer)

['<clr_ctor>',
 'AccrualDays',
 'Accrued',
 'ApproximateForFastCalculation',
 'AsOf',
 'Bond',
 'BreakEvenFee',
 'BreakEvenPremium',
 'BuildPaymentPricer',
 'CDS',
 'Carry',
 'Cashflow',
 'Clone',
 'CollectValidationErrors',
 'ConventionalSpread',
 'Correlation',
 'CounterpartyCurve',
 'CurrentNotional',
 'CurrentRate',
 'DiscountCurve',
 'DiscountingAccrued',
 'EffectiveNotional',
 'Equals',
 'ExpectedLoss',
 'ExpectedLossRate',
 'Fee',
 'FeePv',
 'FeeSettle',
 'FindAccrualStart',
 'FlatFeePv',
 'FlatPrice',
 'FullFeePv',
 'FullModelPrice',
 'FwdPremium',
 'FwdPremium01',
 'FxCorrelation',
 'FxCurve',
 'FxDevaluation',
 'FxVolatility',
 'GenerateCashflow',
 'GetDerivativesWrtOrdinates',
 'GetHashCode',
 'GetPaymentSchedule',
 'GetType',
 'HasAnalyticDerivatives',
 'ISDAConverterDate',
 'ISDADiscountCurve',
 'ISDARecoveryRate',
 'ImpliedCounterpartyCurve',
 'ImpliedDiscountSpread',
 'ImpliedFlatCDSCurve',
 'ImpliedHazardRateSpread',
 'IncludeMaturityAccrual',
 'IncludeMaturityProtectio

In [17]:
sn_pricer.ExpectedLoss()

-322230.5815642844

In [30]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

today = datetime.today()
six_months_later = today + relativedelta(months=6)


In [ ]:
roll_as_of = six_months_later
roll_settle = six_months_later + relativedelta(days=1)
Rolldown(
    pricer=cdx_pricer,
    measure='MarketValue',
    rollAsOf=roll_as_of,
    rollSettle=roll_settle,
    clean=True
)

1451973.412046913

In [32]:
Rolldown(
    pricer=cdx_pricer,
    measure='MarketValue',
    rollAsOf=roll_as_of,
    rollSettle=roll_settle,
    clean=False
)

1495741.6787135806

In [7]:


print('              Inst:', cdx_pricer.CDX)
print('             Quote:', cdx_pricer.MarketQuote * 10000)
print('    Discount Curve:', cdx_pricer.DiscountCurve.Name)
print('    Survival Curve:', cdx_pricer.SurvivalCurves[0])
print('Scaling:', cdx_pricer.Scaling)
# print()
# print(cdx_pricer.DiscountCurve)
# print(cdx_pricer.SurvivalCurve)

              Inst: ITRAXX-XOVERS39V3-5Y
             Quote: 197.722
    Discount Curve: EURIBOR
    Survival Curve: iTraxx Europe Crossover Series 39 Version 3/PICABON.EUR.SeniorUnsecured.ModifiedModified14[SurvivalFitCalibrator]
 name_ = iTraxx Europe Crossover Series 39 Version 3/PICABON.EUR.SeniorUnsecured.ModifiedModified14
 category_ = None
 asOf_ = 10-Jun-2025
 spread_ = 0
 dc_ = Actual/365 Fixed
 freq_ = Continuous
 ccy_ = EUR
 interp_ = Quantifi.Toolkit.Numerics.Weighted
 flags_ = 0
 jumpDate_ = <null>
 dt_[0] = 23-Jun-2026 ; y_[0] = 0.98694132
 dt_[1] = 22-Jun-2027 ; y_[1] = 0.94900644
 dt_[2] = 21-Jun-2028 ; y_[2] = 0.8963012
 dt_[3] = 21-Jun-2030 ; y_[3] = 0.80435177
 dt_[4] = 22-Jun-2032 ; y_[4] = 0.65867019
 dt_[5] = 21-Jun-2035 ; y_[5] = 0.51324141
tenors_.Count = 6; Tenor list:
tenors_[0]: Name = 1 Years; Product = 1 Years; MarketPv = 0; Quote = 0.00755239204510351, CreditSpread; modelPv_ = 0; weight_ = 1
tenors_[1]: Name = 2 Years; Product = 2 Years; MarketPv = 0; Quot

In [11]:
cdx_pricer.IntrinsicValue()

-9860890.132545074

We can now use the pricer object loaded above to calculate some risk measures for the bond:

In [45]:
def print_pr(pricer):
    #print('              Bond:', pricer.Bond)  
    print('             Quote:', pricer.MarketQuote * 100)
    print('           Accrued: {0:,.2f}'.format(pricer.Accrued()))
    # print('         PV from Q: {0:,.2f}'.format(pricer.ProductPvFromQuote()))  
    # print('               YTM: {0:.2%}'.format(pricer.YieldToMaturity()))
    # print('      Mod Duration: {0:.2f}'.format(pricer.ModDuration()))
    # print('Implied CDS Spread: {0:.2f}'.format(pricer.ImpliedCDSLevel() * 10000))

print_pr(cdx_pricer)

             Quote: 104.06064564998529
           Accrued: -175,000.00


Next, we show that we can manipulate the embedded data (in this case, a Market Quote) to run scenarios on the loaded bond pricer: 

In [9]:
dir(cdx_pricer)

type(cdx_pricer)

Quantifi.Toolkit.Pricers.CDXPricer

In [46]:
cdx_pricer.QuotingConvention = QuotingConvention.FlatPrice
cdx_pricer.MarketQuote = 1.08 # no method to re-build pricer, although cheeck cds scalinbg factors if they are recomputed or not

print_pr(cdx_pricer)

             Quote: 108.0
           Accrued: -175,000.00


In the following cell, we change change the yield of the bond, as well as use different discount and survival curves for pricing:

In [47]:

# new_discount_curve = Connect.get_discount_curve(as_of = '2025-04-29', curve_name = 'FedFunds', calc_env='Official', verify_https_cert = False)

new_credit_curve = Connect.get_survival_curve(as_of = '2025-04-29', curve_name = 'WHR.USD.SeniorUnsecured.Excluded14', calc_env = 'Official', verify_https_cert = False)



In [48]:
## Examples of ways to modify a bond pricer object's attributes

# bond_pricer.QuotingConvention = QuotingConvention.Yield
# bond_pricer.MarketQuote = 0.05
# bond_pricer.DiscountCurve = new_discount_curve
# bond_pricer.SurvivalCurve = new_credit_curve

# print_pr(bond_pricer)

***

### Load a Portfolio from the Database

Next section loads portfolios of bond trades (or more specifcally *trade pricer* obects) from The Quantifi Risk system, based on a grouping defined by 'Strategy'. Quantifi pricer objects contain the pricing models which compute various risk measures (such as PVs, greeks, etc) for the portfolio of trades.

First, we load the list of trades in the chosen *Strategy*:


In [49]:
# Load portfolio of bond trades for a given strategy. Note that default values of max_results = 500
hql_query = "FROM Trade b WHERE b.Strategy.Name = 'Manual'"
trade_ptf = Connect.data_query(hql_query, max_results = 5000, verify_https_cert = False)
#display(bond_ptf)

# Extract trade IDs, store in a list
trade_ids = [trade['TradeId'] for trade in trade_ptf]
print('Number of trades:', len(trade_ids))
display(trade_ids)


Number of trades: 4564


['Bond0020145',
 'Bond0020160',
 'Bond0020161',
 'Bond0020202',
 'Bond0020578',
 'Bond0022202',
 'Bond0022234',
 'Bond0022271',
 'Bond0022272',
 'Bond0022273',
 'Bond0022274',
 'Bond0022307',
 'Bond0022429',
 'Bond0022445',
 'Bond0022452',
 'Bond0022484',
 'Bond0022526',
 'CDS0017434',
 'CDS0017647',
 'CDS0018432',
 'CDS0018433',
 'CDS0019255',
 'CDS0019288',
 'CDS0019289',
 'CDS0019886',
 'CDS0020310',
 'CDS0020339',
 'CDS0020396',
 'CDS0020838',
 'CDS0020839',
 'CDS0020961',
 'CDS0020962',
 'CDS0020963',
 'CDS0020970',
 'CDS0020971',
 'CDS0021112',
 'CDS0021382',
 'CDS0021383',
 'CDS0021498',
 'CDS0021678',
 'CDS0021755',
 'CDS0021760',
 'CDS0022190',
 'CDS0022241',
 'CDS0022303',
 'CDS0022364',
 'CDS0022411',
 'CDS0022417',
 'CDS0022418',
 'CDS0022419',
 'CDS0022441',
 'CDS0022454',
 'CDS0022494',
 'CDS0022527',
 'CDO0018074',
 'CDO0019481',
 'CDO0019795',
 'CDO0020164',
 'CDO0020266',
 'CDO0020536',
 'CDO0020960',
 'CDO0021084',
 'CDO0021087',
 'CDO0021445',
 'CDO0022085',
 'CDO002

Now, we can load the pricers for selected trades (load the first 100):

In [69]:
tic = time.time()

ids_to_query = trade_ids

# Load pricers corresponding to the list of trade IDs (they are returned in a dictionary indexed by tradeId). Loading the first 100 pricers.
trade_pricers = Connect.get_pricers_by_trade_ids(as_of='2025-04-29', trade_ids=ids_to_query, calc_env='Official', verify_https_cert = False)  # get_pricers_by_trade_ids() important

toc = time.time()
print(round(toc-tic, 2), 'sec total')
print(round((toc-tic)/len(ids_to_query), 2), 'sec per trade')
print()

398.5 sec total
0.09 sec per trade



Keep only trades that are LIVE/non-terminated

In [70]:
type(trade_pricers)

dict

In [71]:
live_count=0
live_ptf = {}
for id, pricer in trade_pricers.items():
    pprint.pprint(id)
    pprint.pprint(pricer.IsTerminated)
    if not pricer.IsTerminated:
        live_ptf[id] = pricer  # Add to dictionary
        live_count+=1

live_ptf

'CDX0017132'
True
'CDX0017744'
True
'CDX0018557'
True
'CDX0019774'
True
'CDX0022185'
False
'CDX0022331'
True
'CDXOption0017039'
True
'CDXOption0017972'
True
'CDXTranche0015580'
True
'CDXTranche0019456'
True
'CDX0014263'
True
'CDX0017221'
True
'CDX0018001'
True
'CDX0018221'
True
'CDX0018601'
True
'CDX0019359'
True
'CDX0021740'
True
'CDX0021847'
True
'CDXOption0020376'
True
'CDXTranche0017289'
True
'CDX0015483'
True
'CDX0016262'
True
'CDX0016471'
True
'CDX0017340'
True
'CDX0017735'
True
'CDX0017925'
True
'CDX0018377'
True
'CDXOption0017046'
True
'CDXTranche0017817'
True
'CDXTranche0017898'
True
'CDS0022364'
False
'CDX0014046'
True
'CDX0014148'
True
'CDX0014323'
True
'CDX0014484'
True
'CDX0018247'
True
'CDX0021114'
True
'CDX0021476'
True
'CDXOption0018149'
True
'CDXOption0020977'
True
'CDX0014211'
True
'CDX0015229'
True
'CDX0017299'
True
'CDX0019214'
True
'CDX0019710'
True
'CDX0019726'
True
'CDXOption0014182'
True
'CDXOption0014654'
True
'CDXOption0020857'
True
'CDXOption0020904'
True
'CD

{'CDX0022185': <CLR: Quantifi.Toolkit.Pricers.CDXPricer>,
 'CDS0022364': <CLR: Quantifi.Toolkit.Pricers.CDSCashflowPricer>,
 'CDX0022314': <CLR: Quantifi.Toolkit.Pricers.CDXPricer>,
 'CDX0022055': <CLR: Quantifi.Toolkit.Pricers.CDXPricer>,
 'CDX0022421': <CLR: Quantifi.Toolkit.Pricers.CDXPricer>,
 'CDX0022403': <CLR: Quantifi.Toolkit.Pricers.CDXPricer>,
 'CDXTranche0021877': <CLR: Quantifi.Toolkit.Pricers.SyntheticCDOPricer>,
 'CDXTranche0021446': <CLR: Quantifi.Toolkit.Pricers.SyntheticCDOPricer>,
 'CDX0022112': <CLR: Quantifi.Toolkit.Pricers.CDXPricer>,
 'CDX0021912': <CLR: Quantifi.Toolkit.Pricers.CDXPricer>,
 'CDX0022533': <CLR: Quantifi.Toolkit.Pricers.CDXPricer>,
 'CDS0022494': <CLR: Quantifi.Toolkit.Pricers.CDSCashflowPricer>,
 'CDX0022294': <CLR: Quantifi.Toolkit.Pricers.CDXPricer>,
 'CDX0022481': <CLR: Quantifi.Toolkit.Pricers.CDXPricer>,
 'CDX0022451': <CLR: Quantifi.Toolkit.Pricers.CDXPricer>,
 'CDX0022422': <CLR: Quantifi.Toolkit.Pricers.CDXPricer>,
 'CDX0021688': <CLR: Qua

In [72]:
dir(live_ptf['CDXTranche0021877'])

['<clr_ctor>',
 'AccrualDays',
 'AccrualToDate',
 'Accrued',
 'AccruedFractionOnDefault',
 'AdjustDurationForRemainingNotional',
 'ApproximateForFastCalculation',
 'AsOf',
 'Basket',
 'BreakEvenFee',
 'BreakEvenPremium',
 'BuildPaymentPricer',
 'CDO',
 'Carry',
 'Clone',
 'CollectValidationErrors',
 'Copula',
 'Correlation',
 'CounterpartySurvivalCurve',
 'CurrentAttachment',
 'CurrentDetachment',
 'CurrentNotional',
 'CurrentRate',
 'DefaultTiming',
 'DiscountCurve',
 'DiscountCurves',
 'EffectiveNotional',
 'Equals',
 'ExpectedLoss',
 'ExpectedPrepayment',
 'ExpectedSurvival',
 'FeePv',
 'FeePvDerivatives',
 'FeeToDate',
 'FlatFeePv',
 'FlatPrice',
 'ForwardExpectedLoss',
 'FullFeePv',
 'FullPrice',
 'GenerateCashflow',
 'GetDerivativesWrtOrdinates',
 'GetHashCode',
 'GetPaymentSchedule',
 'GetType',
 'HasAnalyticDerivatives',
 'IgnoreAccruedSetting',
 'ImpliedVolatility',
 'IsTerminated',
 'LossToDate',
 'MTMCarry',
 'Maturity',
 'Name',
 'Notional',
 'NotionalToDate',
 'Overloads',

In [90]:
itrx_mezz_3_6_pricer = live_ptf['CDXTranche0021877']

carry = itrx_mezz_3_6_pricer.Carry()


cpn_carry = itrx_mezz_3_6_pricer.CurrentNotional * 
cpn_carry

70000000.0

In [97]:
itrx_mezz_3_6_pricer.BreakEvenFee()#the upfront

0.050325441659750565

In [101]:
itrx_mezz_3_6_pricer.Premium01()# the dv01

31375.64560748404

In [74]:
type(trade_pricers)#dict
trade_pricer = trade_pricers['CDXOption0014182']
#pprint.pprint(dir(trade_pricer))
pprint.pprint(dir(trade_pricer.CDXOption))
pprint.pprint(trade_pricer.CDXOption.Expiration)

['<clr_ctor>',
 'BDConvention',
 'BarrierEnd',
 'BarrierMonitoringFrequency',
 'BarrierPayoffTime',
 'BarrierStart',
 'Barriers',
 'BuyIsNegative',
 'CDX',
 'Calendar',
 'Ccy',
 'Clone',
 'CollectValidationErrors',
 'CustomFirstPeriodAccrualStartDate',
 'CustomPaymentSchedule',
 'DayCount',
 'Description',
 'Effective',
 'EffectiveMaturity',
 'Equals',
 'Expiration',
 'Freq',
 'GetHashCode',
 'GetType',
 'Index',
 'IsActive',
 'IsBarrier',
 'IsDigital',
 'IsDoubleBarrier',
 'IsTouchOption',
 'Issue',
 'Maturity',
 'Name',
 'Notional',
 'Overloads',
 'PaymentLag',
 'PayoffType',
 'Premium',
 'QuoteInPrice',
 'Rebate',
 'Recovery',
 'ReferenceEquals',
 'Series',
 'SettlementType',
 'ShallowCopy',
 'Strike',
 'StrikeDetermination',
 'StrikeIsPrice',
 'Style',
 'Tenor',
 'ToString',
 'TryValidate',
 'Type',
 'Underlying',
 'UnderlyingDetermination',
 'Validate',
 '__class__',
 '__clr_handle_object__',
 '__clr_handle_value__',
 '__clr_type__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__do

Next, we compute a set of risk measures for each bond and store in a Pandas dataframe:

In [75]:
def create_results_df(bond_pricers):

    Bond_trades_df = pd.DataFrame(index = bond_pricers.keys())  

    # iterate trhough each row and calculate risks for each trade
    for name, pricer in bond_pricers.items():

        pricer.Notional = 10000000

       # Bond_trades_df.at[name, 'Bond Name'] = pricer.Bond
        Bond_trades_df.at[name, 'Notional'] = pricer.Notional
       # Bond_trades_df.at[name, 'Dirty Price'] = pricer.FullPrice() * 100     
        Bond_trades_df.at[name, 'Accrued'] = pricer.Accrued() 
        # Bond_trades_df.at[name, 'Yield'] = pricer.YieldToMaturity()       
        # Bond_trades_df.at[name, 'PV01'] = pricer.PV01() * 10000
        # Bond_trades_df.at[name, 'Yield Convexity'] = pricer.Convexity()
        # Bond_trades_df.at[name, 'Mod Duration'] = pricer.ModDuration()
        Bond_trades_df.at[name, 'Pv'] = pricer.Pv()
        # Bond_trades_df.at[name, 'Z-Spread'] = pricer.ImpliedZSpread() * 10000
        # Bond_trades_df.at[name, 'Implied CDS Level'] = pricer.ImpliedCDSLevel() * 10000

    return Bond_trades_df

def print_results(bond_trade_df):
    # Format and print out results
    format_dict = {}
    for col_name in [ 'Accrued', 'Pv', 'Notional',]: format_dict[col_name] = '{0:,.2f}'
    #for col_name in [ 'Yield' ]: format_dict[col_name] = '{0:.2%}'
    #for col_name in [ 'Quote', 'Dirty Price', 'Mod Duration', 'Z-Spread Duration', 'IR Duration', 'Spread Duration', 'Spread Convexity', 'Yield Convexity', 'PV01', 
    #                  'Implied CDS Level', 'Implied CDS Basis', 'Model Dirty Price', 'Z-Spread', 'ASW Spread', 'Rate Convexity']: format_dict[col_name] = '{0:.2f}'
    format_dict['Maturity'] = '{:%d-%m-%y}'

    df_style = bond_trade_df.style.format(format_dict)

    total_pv = bond_trade_df['Pv'].sum()
    print('Total PV: {0:,.2f}'.format(total_pv))

    display(df_style)

results_df = create_results_df(bond_pricers)
print_results(results_df)

NameError: name 'bond_pricers' is not defined

***
We can compute sensitivities for our portfolio of bonds using Quantifi's sensitivity calculators. As an example, here we compute IR sensitivity for the above portfolio:

In [24]:
# Rates Sensitivty calculator
tic = time.time()

results_table = Sensitivities.Rate(pricers = list(bond_pricers.values()), 
                                   measure = 'Pv', 
                                   initialBump = 0.0, 
                                   upBump = 10.0, 
                                   downBump = 0, 
                                   bumpRelative = False, 
                                   scaledDelta = True,
                                   bumpType = BumpType.ByTenor, 
                                   bumpTenors = None, 
                                   calcGamma = False, 
                                   calcHedge = False, 
                                   hedgeTenor = None, 
                                   recalibrate = False, 
                                   dataTable = None, 
                                   curveFitmethod = None,  
                                   rescaleStrikes = None, 
                                   discountInterp = None)
toc = time.time()

#set up a pandas dataframe to store sensitivities (for ease of use)
df = results_table.to_dataframe()

print('Time, s:', round(toc-tic, 2))

# Print out sensitivities
display(df.pivot_table(index='Curve Tenor', columns='Element', values='Delta', aggfunc=np.sum, margins=True, margins_name='Total').drop(columns = ['Total']).style.format('{:,.0f}'))


Time, s: 1.11


C:\Users\mmoore\AppData\Local\Temp\ipykernel_42596\1850209799.py:29: FutureWarning: The provided callable <function sum at 0x000001B5F9E76020> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  display(df.pivot_table(index='Curve Tenor', columns='Element', values='Delta', aggfunc=np.sum, margins=True, margins_name='Total').drop(columns = ['Total']).style.format('{:,.0f}'))
C:\Users\mmoore\AppData\Local\Temp\ipykernel_42596\1850209799.py:29: FutureWarning: The provided callable <function sum at 0x000001B5F9E76020> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  display(df.pivot_table(index='Curve Tenor', columns='Element', values='Delta', aggfunc=np.sum, margins=True, margins_name='Total').drop(columns = ['Total']).style.format('{:,.0f}'))


Element,EURIBOR
Curve Tenor,
EURIBOR_6 Months.MM_1 Months,5
EURIBOR_6 Months.MM_3 Months,20
EURIBOR_6 Months.MM_6 Months,7
EURIBOR_6 Months.Swap_1 Years,0
EURIBOR_6 Months.Swap_2 Years,0
EURIBOR_6 Months.Swap_3 Years,0
EURIBOR_6 Months.Swap_4 Years,0
EURIBOR_6 Months.Swap_5 Years,0
EURIBOR_6 Months.Swap_6 Years,0


***
### FX Examples

In the following example, we load a set of pricers for FX Forwards and options. We don't need to explicitly set them up, loaded pricers will contain all relevant information and objecyts representing the trades will be of a correct type for each product.

In [ ]:
# Load an FX Forward trade
pricer = Connect.get_pricer_by_trade_id(as_of = '2019-05-09', 
                                        trade_id = 'FxForward0024431', 
                                        calc_env = 'Official', 
                                        verify_https_cert = False)

print('Product:')
print(pricer.FxForward.Description)
print(pricer.FxForward.Maturity)
print(str(pricer.FxForward.PayCcy) + '/' + str(pricer.FxForward.ReceiveCcy))

print('\nPricing:')
print("PV        :", pricer.Pv())
print("SpotFX    :", pricer.SpotFxRate(pricer.FxForward.PayCcy, pricer.FxForward.ReceiveCcy))
print("PayAmount :", pricer.PayAmount())
print("Fwd points:", pricer.ForwardFxPoints(pricer.FxForward.PayCcy, pricer.FxForward.ReceiveCcy))



In [ ]:
# Load an FX option trade
pricer = Connect.get_pricer_by_trade_id(as_of = '2019-05-09', 
                                        trade_id = 'FxOption0023021', 
                                        calc_env = 'Official', 
                                        verify_https_cert = False)
print('Product:')
print(pricer.FxOption)
print(pricer.FxOption.Type)
print(pricer.FxOption.Style)
print(pricer.FxOption.Maturity)

print('\nPricing:')
print("PV    :", pricer.Pv())
print("Delta :", pricer.Delta())
print("Gamma :", pricer.Gamma())
print("Vega  :", pricer.Vega())


In [ ]:
# Load portfolio of FX Forwards for a given strategy. Note that default values of max_results = 500
hql_query = "FROM Trade b WHERE b.TradeId LIKE 'FXForward%'"
options_ptf = Connect.data_query(hql_query, verify_https_cert = False)

# Extract trade IDs, store in a list
trade_ids = [trade['TradeId'] for trade in options_ptf]
print('Number of trades:', len(trade_ids))

pricers = Connect.get_pricers_by_trade_ids(as_of = '2019-05-09', 
                                           trade_ids = trade_ids, 
                                           calc_env = 'Official', 
                                           verify_https_cert = False)

trades_df = pd.DataFrame(index = trade_ids)
for tradeId, pricer in pricers.items():
    trades_df.at[tradeId, 'Product'] = str(pricer.FxForward.PayCcy) + '/' + str(pricer.FxForward.ReceiveCcy)
    trades_df.at[tradeId, 'Maturity'] = pricer.FxForward.Maturity
    trades_df.at[tradeId, 'Fwd Points'] = pricer.ForwardFxPoints(pricer.FxForward.PayCcy, pricer.FxForward.ReceiveCcy)
    trades_df.at[tradeId, 'PV'] = pricer.Pv()

display(trades_df.style.format({'PV':'{:,.2f}', 'Fwd Points':'{:.0f}', 'Maturity':'{:%d-%m-%Y}'}))

***

### Equity Products Examples

In the following example, we load a set of pricers for a single option and a portfolio of options. We don't need to explicitly set them up, loaded pricers will contain all relevant information and objecyts representing the trades will be of a correct type for each product.

In [ ]:
# Load a equity option trade
pricer = Connect.get_pricer_by_trade_id(as_of = '2019-05-09', 
                                        trade_id = 'StockOption24315', 
                                        calc_env = 'Official', 
                                        verify_https_cert = False)
print('Product')
print(pricer.StockOption)
print(pricer.StockOption.Type)
print(pricer.StockOption.Style)
print(pricer.StockOption.Maturity)

print('\nStock Curve')
display(pricer.StockCurve)

print('\nPricing')
print("Vol:    ", pricer.Volatility)
print("PV:     ", pricer.Pv())
print("Delta:  ", pricer.Delta())
print("Gamma:  ", pricer.Gamma())
print("Vega:   ", pricer.Vega())
print("Rho:    ", pricer.Rho())


***

**Option portfolio:**

In [ ]:
# Load portfolio of bond trades for a given strategy. Note that default values of max_results = 500
hql_query = "FROM Trade b WHERE b.TradeId LIKE 'StockOption%'"
options_ptf = Connect.data_query(hql_query, verify_https_cert = False)

# Extract trade IDs, store in a list
trade_ids = [trade['TradeId'] for trade in options_ptf]
print('Number of trades:', len(trade_ids))

pricers = Connect.get_pricers_by_trade_ids(as_of = '2019-05-09', 
                                           trade_ids = trade_ids, 
                                           calc_env = 'Official', 
                                           verify_https_cert = False)

trades_df = pd.DataFrame(index = trade_ids)
for tradeId, pricer in pricers.items():
    trades_df.at[tradeId, 'Product'] = pricer.Product
    trades_df.at[tradeId, 'PV'] = pricer.Pv()
    trades_df.at[tradeId, 'Delta'] = pricer.Delta()
    trades_df.at[tradeId, 'Gamma'] = pricer.Gamma()
    trades_df.at[tradeId, 'Vega'] = pricer.Vega()
    trades_df.at[tradeId, 'Rho'] = pricer.Rho()

display(trades_df.style.format({'PV':'{:,.2f}'}))

In [ ]:
from Quantifi.Toolkit.Base import Dt
from Quantifi.Toolkit.Sensitivity import ThetaFlags, SensitivityRescaleStrikes

thetas = Sensitivities.Theta(pricers = list(pricers.values()), 
                             measure = 'Pv', 
                             toAsOf = Dt(20190510), 
                             toSettle = Dt(20190511), 
                             flags = ThetaFlags.Recalibrate, 
                             rs = [SensitivityRescaleStrikes.UsePricerSetting]*len(pricers))

#set up a pandas dataframe to store sensitivities (for ease of use)
df = pd.DataFrame({'Pricer':pricers.keys(), 'Theta':thetas})

# Print out thetas
display(df.style.format({'Theta':'{:,.2f}'}))


***
### Commodity Examples

In the following examples, we load a set of pricers for Commodity Futures and Listed Future Options. We don't need to explicitly set them up, loaded pricers will contain all relevant information and objecyts representing the trades will be of a correct type for each product.

In [ ]:
# Load a Commodity Forward trade
pricer = Connect.get_pricer_by_trade_id(as_of = '2019-05-09', 
                                        trade_id = 'CommodityFuture0025678', 
                                        calc_env = 'Official', 
                                        verify_https_cert = False)
print("Currency: ", pricer.Product.Ccy)
print("Maturity: ", pricer.Product.EffectiveMaturity)
print("Quote:    ", pricer.QuotedPrice)
print("Notional: ", pricer.Notional)
print("PV:       ", pricer.Pv())

In [ ]:
# Load a Commodity Option trade
pricer = Connect.get_pricer_by_trade_id(as_of = '2019-05-09', 
                                        trade_id = 'CommodFutListOpt0025083', 
                                        calc_env = 'Official', 
                                        verify_https_cert = False)
print('Product:')
print("Option:   ", pricer.CommodityFutureOption)
print("Expiry:   ", pricer.Product.EffectiveMaturity)
print("Notional: ", pricer.Notional)

print('\nCommodity Curve')
display(pricer.CommodityCurve)

print('\nPricing:')
print("PV:    ", pricer.Pv())
print("Delta: ", pricer.Delta())
print("Gamma: ", pricer.Gamma())
print("Vega:  ", pricer.Vega())
print("Charm: ", pricer.Charm())

***

**Listed Commodity Future Option Portfolio:**

In [ ]:
trade_ids = ['CommodFutListOpt0025082', 'CommodFutListOpt0025083', 'CommodFutListOpt0025087', 'CommodFutListOpt0025088']
pricers = Connect.get_pricers_by_trade_ids(as_of = '2019-05-09', 
                                           trade_ids = trade_ids, 
                                           calc_env = 'Official', 
                                           verify_https_cert = False)

trades_df = pd.DataFrame(index = trade_ids)
for tradeId, pricer in pricers.items():
    trades_df.at[tradeId, 'Product'] = pricer.CommodityFutureOption
    trades_df.at[tradeId, 'Expiry'] = pricer.Product.EffectiveMaturity 
    trades_df.at[tradeId, 'PV'] = pricer.Pv()
    trades_df.at[tradeId, 'Delta'] = pricer.Delta()
    trades_df.at[tradeId, 'Gamma'] = pricer.Gamma()
    trades_df.at[tradeId, 'Vega'] = pricer.Vega()
    trades_df.at[tradeId, 'Charm'] = pricer.Charm()

display(trades_df.style.format({'PV':'{:,.2f}', 'Delta':'{:,.2f}', 'Gamma':'{:,.2f}', 'Vega':'{:,.2f}', 'Charm':'{:,.2f}', 'Expiry':'{:%d-%m-%Y}'}))

***

### Load Pricers for Mixed Products

In the following example, we load a set of pricers for a diverse set of products (Bonds, Swaps, CDS, Credit Index). We don't need to explicitly set them up, loaded pricers will contain all relevant information and objecyts representing the trades will be of a correct type for each product.

In [ ]:
trade_ids = ['Bond0024393','Bond0024408','Bond0024406','Bond0024364','Bond0024402',
             'CDX0024485', 'CDX0024471',
             'CDS0024932', 'CDS0024931', 'CDS0024914',
             'Swap0020807', 'Swap0024436', 'Swap0025485',
             'StockOption24311', 'StockOption24316', 'StockOption24319', 'StockOption24320',
             'FxForward0024447', 'FxForward0024423', 'FxOption0023021', 'FxOption0023017']

pricers = Connect.get_pricers_by_trade_ids(as_of = '2019-05-09', 
                                           trade_ids = trade_ids, 
                                           calc_env = 'Official', 
                                           verify_https_cert = False)

trades_df = pd.DataFrame(index = trade_ids)
for tradeId, pricer in pricers.items():
    trades_df.at[tradeId, 'Product'] = pricer.Product
    trades_df.at[tradeId, 'PV'] = pricer.Pv()

display(trades_df.style.format({'PV':'{:,.2f}'}))

We can now calculate aggregated sensitivities for the combined portfolio of trades. Using bucketed spread sensitivity as an example:

In [25]:
# Spread Sensitivity calculator
results_table = Sensitivities.Spread(pricers = list(pricers.values()), 
                                     measure = 'Pv', 
                                     targetQuoteType = QuotingConvention.CreditSpread, 
                                     initialBump = 0.0, 
                                     upBump = 1.0, 
                                     downBump = 0, 
                                     bumpRelative = False, 
                                     scaledDelta = True,
                                     bumpType = BumpType.ByTenor, 
                                     bumpTenors = None, 
                                     calcGamma = True, 
                                     calcHedge = False, 
                                     hedgeTenor = None, 
                                     sensitivityMethod = SensitivityMethod.FiniteDifference,
                                     dataTable = None, 
                                     rescaleStrikes = None)

#set up a pandas dataframe to store sensitivities (for ease of use)
df = results_table.to_dataframe()

# Print out sensitivities
display(df.pivot_table(index='Element', columns='Curve Tenor', values='Delta', aggfunc=np.sum).style.format('{:,.0f}'))

NameError: name 'pricers' is not defined

***
## "Complex" products

Load pricers for 'complex' structured credit products such as CDOs, index tranches and index trades with 100+ underlying credits. 

In [28]:
# Load a set of complex pricers
trade_ids = ['CDXTranche0021876',]

pricers = Connect.get_pricers_by_trade_ids(as_of = '2025-03-04', 
                                           trade_ids = trade_ids, 
                                           calc_env = 'Official', 
                                           verify_https_cert = False)

trades_df = pd.DataFrame(index = trade_ids)
for tradeId, pricer in pricers.items():
    trades_df.at[tradeId, 'Product'] = pricer.Product
    trades_df.at[tradeId, 'PV'] = pricer.Pv()

display(trades_df.style.format({'PV':'{:,.2f}'}))

print("\nCDO CDO0024669 Survival Curves:")
display(list(pricers["CDXTranche0021876"].SurvivalCurves))

,Product,PV
CDXTranche0021876,CDXTranche0021876,"2,161,621.73"



CDO CDO0024669 Survival Curves:


[<CLR: iTraxx Europe Series 36 Version 1/EDF.EUR.SeniorUnsecured.ModifiedModified14[SurvivalFitCalibrator]
  name_ = iTraxx Europe Series 36 Version 1/EDF.EUR.SeniorUnsecured.ModifiedModified14
  category_ = None
  asOf_ = 04-Mar-2025
  spread_ = 0
  dc_ = Actual/365 Fixed
  freq_ = Continuous
  ccy_ = EUR
  interp_ = Quantifi.Toolkit.Numerics.Weighted
  flags_ = 0
  jumpDate_ = <null>
  dt_[0] = 23-Dec-2025 ; y_[0] = 0.99682139
  dt_[1] = 22-Dec-2026 ; y_[1] = 0.99102657
  dt_[2] = 21-Dec-2027 ; y_[2] = 0.98302671
  dt_[3] = 21-Dec-2029 ; y_[3] = 0.95153914
  dt_[4] = 23-Dec-2031 ; y_[4] = 0.90738373
  dt_[5] = 21-Dec-2034 ; y_[5] = 0.81324036
 tenors_.Count = 6; Tenor list:
 tenors_[0]: Name = 1 Years; Product = 1 Years; MarketPv = 0; Quote = 0.00234568959438184, CreditSpread; modelPv_ = 0; weight_ = 1
 tenors_[1]: Name = 2 Years; Product = 2 Years; MarketPv = 0; Quote = 0.00296064500287428, CreditSpread; modelPv_ = 0; weight_ = 1
 tenors_[2]: Name = 3 Years; Product = 3 Years; Marke